# Pilot · Step 5 only (evaluation on val), with the probes of a finished pilot_03 run

Runs step 5 (every probe on the 50 val scenes, official ScanNet++ metric, cross-check with the toolbox's eval_semantic) from the `probes.pt` that `pilot_03_probes` saved, without redoing steps 3 and 4. No `SCANNETPP_TOKEN` needed.

**Before running**
- [ ] **Accelerator:** GPU T4 x2
- [ ] **Internet:** on
- [ ] **Secrets** (Add-ons → Secrets): `GITHUB_TOKEN` attached
- [ ] **Environment:** pin to the same image as the wheelhouse (Settings → Environment preference); install_env.py refuses a mismatched image
- [ ] **Input:** private dataset `appearance-pt-wheels` (output of m1_01)
- [ ] **Input:** private dataset `scannetpp-core-val`
- [ ] **Input:** `probes.pt` from a finished `pilot_03_probes` run (its `pilot_concat/step4/` folder): attach that notebook's output (Add Input → Your Work → Notebooks), or upload the file as a private dataset and attach it

**Run:** Save Version → Save & Run All (Commit) (expected about 45–60 min).

**Paste back:** the SUMMARY block of step 5 (full log: /kaggle/working/logs/pilot_04_step5.log; results: /kaggle/working/pilot_concat/step5/results.json).

*Generated by `scripts/make_notebooks.py`; edit the scripts, not this notebook.*

In [ ]:
# ---- run configuration: the only cell to edit ----
REPO = "eylulpelinkilic/appearance-pt"  # your private GitHub repo (owner/name)
REF = "main"                  # branch, tag or commit SHA; the resolved commit is logged

In [ ]:
"""Kaggle bootstrap: fetch this (private) repo at a fixed ref, then run entry points with logs.

Embedded verbatim in every notebook by scripts/make_notebooks.py; edit here, then regenerate.
The GitHub token comes from Kaggle Secrets and reaches git only through environment variables
(never argv, files or output), so it cannot leak into logs or tracebacks; git's error messages
are shown (a notebook would otherwise hide them) with the token redacted.
"""

import base64
import os
import pathlib
import shutil
import subprocess
import sys
import time

_GIT_DEBUG_VARS = ("GIT_TRACE", "GIT_TRACE_CURL", "GIT_CURL_VERBOSE", "GIT_TRACE_PACKET")


def _git(args, cwd=None, env=None, capture=False, redact=()):
    # stderr is captured: a notebook cell never shows a subprocess's stderr, only the exit code
    r = subprocess.run(["git", *args], cwd=cwd, env=env, text=True, capture_output=True)
    if r.returncode != 0:
        err = (r.stderr or "").strip()
        for s in redact:
            err = err.replace(s, "***")
        raise RuntimeError(f"git {args[0]} failed (exit {r.returncode}):\n{err}")
    return r.stdout.strip() if capture else None


def clone_repo(repo, ref, dest="/tmp/appearance-pt", get_secret=None):
    """Fetch `repo` ("owner/name") at `ref` (branch, tag or commit) into `dest`.

    Submodules (public) are fetched at their pinned commits. Returns (dest, commit).
    """
    if repo.startswith("OWNER/"):
        raise ValueError('set REPO in the config cell to your GitHub repo ("owner/name")')
    if get_secret is None:
        from kaggle_secrets import UserSecretsClient

        get_secret = UserSecretsClient().get_secret
    token = get_secret("GITHUB_TOKEN")
    auth = base64.b64encode(f"x-access-token:{token}".encode()).decode()
    env = {k: v for k, v in os.environ.items() if k not in _GIT_DEBUG_VARS}
    env.update(
        GIT_TERMINAL_PROMPT="0",
        GIT_CONFIG_COUNT="1",
        GIT_CONFIG_KEY_0="http.https://github.com/.extraheader",
        GIT_CONFIG_VALUE_0=f"AUTHORIZATION: basic {auth}",
    )
    hide = (token, auth)
    del token, auth
    dest = pathlib.Path(dest)
    if dest.exists():
        shutil.rmtree(dest)
    _git(["init", "-q", str(dest)])
    _git(["remote", "add", "origin", f"https://github.com/{repo}.git"], cwd=dest)
    try:
        _git(["fetch", "-q", "--depth", "1", "origin", ref], cwd=dest, env=env, redact=hide)
    except RuntimeError as e:
        raise RuntimeError(
            f"{e}\n\nCould not fetch {repo} @ {ref}. Check: REPO/REF in the config cell; Internet on; "
            "GITHUB_TOKEN attached to this notebook and allowed to read this repo "
            "(fine-grained token: repo selected under 'Repository access', 'Contents: Read-only', not expired)."
        ) from None
    finally:
        del env, hide
    _git(["checkout", "-q", "FETCH_HEAD"], cwd=dest)
    _git(["submodule", "update", "--init", "--depth", "1", "-q"], cwd=dest)
    commit = _git(["rev-parse", "HEAD"], cwd=dest, capture=True)
    print(f"repo {repo} @ {ref} -> {commit}\n{_git(['submodule', 'status'], cwd=dest, capture=True)}")
    return dest, commit


def run_logged(cmd, log_path, cwd, extra_env=None):
    """Run `cmd`, streaming output to the notebook and appending it to `log_path`.

    Raises if the command fails, so a failed step stops "Save & Run All" visibly.
    """
    log_path = pathlib.Path(log_path)
    log_path.parent.mkdir(parents=True, exist_ok=True)
    env = {**os.environ, "PYTHONUNBUFFERED": "1", **(extra_env or {})}
    t0 = time.time()
    with open(log_path, "a") as log:
        log.write(f"\n### {time.strftime('%Y-%m-%d %H:%M:%S')} $ {' '.join(map(str, cmd))}\n")
        with subprocess.Popen(cmd, cwd=cwd, env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1) as p:
            for line in p.stdout:
                sys.stdout.write(line)
                log.write(line)
            rc = p.wait()
    print(f"\n[exit code {rc} after {(time.time() - t0) / 60:.1f} min; log: {log_path}]")
    if rc != 0:
        raise RuntimeError(f"step failed with exit code {rc}; see {log_path}")


CODE_DIR, COMMIT = clone_repo(REPO, REF)
PYTHONPATH = {"PYTHONPATH": f"{CODE_DIR}/src"}

In [ ]:
# pilot scripts import the Utonia, Sonata and Concerto inference packages (pinned submodules) and our src/
PILOT_PYTHONPATH = {"PYTHONPATH": ":".join(f"{CODE_DIR}/{p}" for p in ("src", "third_party/utonia", "third_party/sonata",
                                                                      "third_party/concerto"))}

In [ ]:
# the step-4 probes: probes.pt from a pilot_03_probes output, or uploaded on its own as a private dataset
import pathlib
found = sorted(pathlib.Path("/kaggle/input").rglob("probes.pt"), key=lambda p: "step4" not in str(p))
assert found, "attach a dataset or notebook output that contains probes.pt (pilot_concat/step4/ of pilot_03_probes)"
PROBES = str(found[0])
print(f"{len(found)} probes.pt found; using {PROBES}")

In [ ]:
run_logged([sys.executable, "-u", "scripts/setup/install_env.py"],
           "/kaggle/working/logs/pilot_04_install.log", cwd=CODE_DIR, extra_env=PYTHONPATH)

In [ ]:
run_logged([sys.executable, "-u", "experiments/pilot_concat/step5_eval.py", "--probes", PROBES],
           "/kaggle/working/logs/pilot_04_step5.log", cwd=CODE_DIR, extra_env=PILOT_PYTHONPATH)